Preparando o ambiente

In [0]:
from PIL import Image
display(Image.open("/Volumes/bikestore/logistics/bikestore_resource/origem/schema.png"))

In [0]:
'''
1º Criar o catálogo bikestore
2º Criar pastas no container
<store-account>/<container>/
|
|- bikestore/
|   |- resource/                # dados de origem e entrega
|   |   |- origem/              # arquivos CSV/Parquet recebidos da fonte
|   |   |- destino/             # exportações (CSV, Parquet, Excel) para cliente ou setores
|   |
|   |- bronze/                  # dados crus (raw)
|   |   |
|   |- silver/                  # dados limpos / tratados
|   |   |
|   |- gold/                    # dados agregadas / analytics

3º Criar o Schema logistics para receber as tabelas e columes
4º Criar volume (bikestore_resource) com a Origem de Dados (deletar todos os anteriores para não ter conflito)
5º Criar Pipeline de dados completo com orquestração
'''


In [0]:
%sql
-- deletando tudo para deixar o ambiente limpo
drop schema if exists bikestore cascade;
drop volume if exists bikestore_resource;
drop catalog if exists bikestore cascade;

In [0]:
%sql
-- Criando o CATALOG "bikestore"
CREATE CATALOG IF NOT EXISTS bikestore;
-- Criando o SCHEMA logistics
CREATE SCHEMA IF NOT EXISTS bikestore.logistics;
-- Criando o VOLUME bikestore_resource
CREATE VOLUME IF NOT EXISTS bikestore.logistics.bikestore_resource;

In [0]:
# criando as pastas de destino e de origem, dentro do bikestore_resource
# dbutils.fs.mkdirs('/Volumes/bikestore/logistics/bikestore_resource/origem/')
# dbutils.fs.mkdirs('/Volumes/bikestore/logistics/bikestore_resource/destino/')

# depois subindo todos os csv na pasta destino

# criando as pastas das camadas bronze, silver e gold
# dbutils.fs.mkdirs('/Volumes/bikestore/logistics/bikestore_resource/bronze/')
# dbutils.fs.mkdirs('/Volumes/bikestore/logistics/bikestore_resource/silver/')
# dbutils.fs.mkdirs('/Volumes/bikestore/logistics/bikestore_resource/gold/')

# mapeamento do volume
display(dbutils.fs.ls('/Volumes/bikestore/logistics/bikestore_resource/'))

In [0]:
# Definindo as pastas do projeto em variáveis para agilizar o processo

bronze_path   = '/Volumes/bikestore/logistics/bikestore_resource/bronze'
silver_path   = '/Volumes/bikestore/logistics/bikestore_resource/silver'
gold_path     = '/Volumes/bikestore/logistics/bikestore_resource/gold'
resource_path = '/Volumes/bikestore/logistics/bikestore_resource/origem'

In [0]:
display(dbutils.fs.ls(resource_path))

In [0]:
# Lendo o arquivo csv e convertendo em Parquet
df_brands = spark.read.csv(f'{resource_path}/brands.csv',
                           header=True,
                           inferSchema=True,
                           sep=','
                           )

display(df_brands)

In [0]:
# salvando na pasta bronze, em parquet como delta, os arquivos de origem.
df_brands.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .save(f'{bronze_path}/brands')

In [0]:
display(dbutils.fs.ls(f'{bronze_path}/brands'))